In [ ]:
import numpy as np
import torch
import anndata as ad
import pandas as pd
import time
from deg_zinb import fit_glm
from deg_zinb.torch_backend.fit import FitConfig
from deg_zinb.torch_backend.model import GLMConfig

In [ ]:
# adata = ad.read_h5ad("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/Results/cropseq_matrix_v2/obj.h5ad")
# covariates_df = pd.read_csv("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/Results/DEG_Analysis/DEG_covariates.csv", index_col=0)
# gene_target = "FOXP2"
# grna_target = "FOXP2"
# covariates_df["expression"] = adata[:,gene_target].X.toarray().flatten()
# df_sub = covariates_df[(covariates_df['gRNA'].str.contains(grna_target)) | (covariates_df['gRNA'].str.contains("non-targeting"))]
# df_sub["group"] = np.where(df_sub['gRNA'].str.contains(grna_target), 1, 0)
# common = adata.obs_names.intersection(df_sub.index)
# adata_sub = adata[common, :].copy()
# df_sub = df_sub.loc[common].copy()   

In [ ]:
# predictors = ["group", "grna_n_nonzero", "grna_n_umis", "response_n_umis", "percent_mt", "inlet"]
# X = df_sub[predictors].copy()

# log_cols = ["grna_n_umis", "response_n_umis"]
# X[log_cols] = X[log_cols].apply(lambda s: np.log1p(pd.to_numeric(s, errors="coerce")))

# num_cols = ["group", "grna_n_nonzero", "percent_mt"]
# for c in num_cols:
#     X[c] = pd.to_numeric(X[c], errors="coerce")

# X["inlet"] = X["inlet"].astype("category")
# X = pd.get_dummies(X, columns=["inlet"], drop_first=True)

# X.insert(0, "Intercept", 1.0)

# mask = X.notna().all(axis=1)
# adata_sub = adata_sub[mask.values, :].copy()
# X = X.loc[mask].astype(np.float32)
# adata_sub.write_h5ad("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/tests/Results/adata_sub.h5ad")
# X.to_csv("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/tests/Results/X_design.csv", index=False)

In [ ]:
adata_sub = ad.read_h5ad("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/tests/Results/adata_sub.h5ad")
X = pd.read_csv("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/tests/Results/X_design.csv")
gene_target = "FOXP2"
grna_target = "FOXP2"

In [14]:
from deg_zinb import fit_glm
import time
start_time = time.time()
res_adam = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="nb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="adam", lr=5e-4, max_iter=2000, rtol=1e-7, verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_adam = time.time() - start_time
start_time = time.time()
res_hybrid = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="nb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="hybrid", lr=5e-4, max_iter=2000, rtol=1e-7, verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_hybrid = time.time() - start_time
start_time = time.time()
res_lbfgs = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="nb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="lbfgs", lr=5e-4, max_iter=2000, rtol=1e-7, verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_lbfgs = time.time() - start_time
print(f'''
      Elapsed time (Adam): {elapsed_time_adam:.2f} seconds
      Elapsed time (Hybrid): {elapsed_time_hybrid:.2f} seconds
      Elapsed time (L-BFGS): {elapsed_time_lbfgs:.2f} seconds
      ''')


      Elapsed time (Adam): 2.35 seconds
      Elapsed time (Hybrid): 0.26 seconds
      Elapsed time (L-BFGS): 0.20 seconds
      


In [6]:
from deg_zinb import fit_glm
import time
start_time = time.time()
res_adam = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="zinb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="adam", verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_adam = time.time() - start_time
start_time = time.time()
res_hybrid = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="zinb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="hybrid",  verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_hybrid = time.time() - start_time
start_time = time.time()
res_lbfgs = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="zinb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="lbfgs",  verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
res_lbfgs.__getattribute__

elapsed_time_lbfgs = time.time() - start_time
print(f'''
      Elapsed time (Adam): {elapsed_time_adam:.2f} seconds
        Log Likelihood (Adam): { -res_adam[gene_target]._neg_log_likelihood:.2f}
      Elapsed time (Hybrid): {elapsed_time_hybrid:.2f} seconds
        Log Likelihood (Hybrid): { -res_hybrid[gene_target]._neg_log_likelihood:.2f}
      Elapsed time (L-BFGS): {elapsed_time_lbfgs:.2f} seconds
        Log Likelihood (L-BFGS): { -res_lbfgs[gene_target]._neg_log_likelihood:.2f}
''')

/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/src/deg_zinb/inference.py:90: RuntimeWarning: divide by zero encountered in divide
  z = theta_hat / se



      Elapsed time (Adam): 0.21 seconds
        Log Likelihood (Adam): -4423.43
      Elapsed time (Hybrid): 0.69 seconds
        Log Likelihood (Hybrid): -4393.27
      Elapsed time (L-BFGS): 1.03 seconds
        Log Likelihood (L-BFGS): -4390.88



In [7]:
summary_est = pd.concat([res_hybrid["FOXP2"].summary()["Estimate"],
            res_lbfgs["FOXP2"].summary()["Estimate"]],axis=1)
summary_est.columns = ["Hybrid","L-BFGS"]
summary_est.index = [f"count: {col}" for col in X.columns.to_list()]+[f"zero: {col}" for col in X.columns.to_list()]  + ["log_theta"]
summary_est["Difference"] = summary_est["Hybrid"] - summary_est["L-BFGS"]
summary_est

,Hybrid,L-BFGS,Difference
count: Intercept,-4.291608,-4.590323,0.298715
count: group,0.095238,0.070988,0.024250
count: grna_n_nonzero,-0.012718,-0.008237,-0.004481
count: grna_n_umis,0.153669,0.125506,0.028163
count: response_n_umis,0.456944,0.467824,-0.010880
count: percent_mt,-0.472063,-0.451704,-0.020359
count: inlet_inlet2,-0.418316,-0.364385,-0.053931
count: inlet_inlet3,-0.675630,-0.612601,-0.063029
zero: Intercept,1.094702,1.224737,-0.130035
zero: group,1.799887,2.840213,-1.040326


In [8]:
from deg_zinb import fit_glm
import time
start_time = time.time()
res_adam = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="mzinb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="adam", verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_adam = time.time() - start_time
start_time = time.time()
res_hybrid = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="mzinb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="hybrid",  verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)
elapsed_time_hybrid = time.time() - start_time
start_time = time.time()
res_lbfgs = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,                 
    model="mzinb",                 
    offset_key=None,            
    fit_cfg=FitConfig(method="lbfgs",  verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
)

elapsed_time_lbfgs = time.time() - start_time
print(f'''
      Elapsed time (Adam): {elapsed_time_adam:.2f} seconds
        Log Likelihood (Adam): { -res_adam[gene_target]._neg_log_likelihood:.2f}
      Elapsed time (Hybrid): {elapsed_time_hybrid:.2f} seconds
        Log Likelihood (Hybrid): { -res_hybrid[gene_target]._neg_log_likelihood:.2f}
      Elapsed time (L-BFGS): {elapsed_time_lbfgs:.2f} seconds
        Log Likelihood (L-BFGS): { -res_lbfgs[gene_target]._neg_log_likelihood:.2f}
''')


      Elapsed time (Adam): 3.33 seconds
        Log Likelihood (Adam): -4389.32
      Elapsed time (Hybrid): 1.16 seconds
        Log Likelihood (Hybrid): -4387.03
      Elapsed time (L-BFGS): 1.03 seconds
        Log Likelihood (L-BFGS): -4387.23



In [15]:
summary_est = pd.concat([res_hybrid["FOXP2"].summary()["Estimate"],
            res_lbfgs["FOXP2"].summary()["Estimate"]],axis=1)
summary_est.columns = ["Hybrid","L-BFGS"]
# summary_est.index = [f"count: {col}" for col in X.columns.to_list()]+[f"zero: {col}" for col in X.columns.to_list()]  + ["log_theta"]
summary_est["Difference"] = summary_est["Hybrid"] - summary_est["L-BFGS"]
summary_est

,Hybrid,L-BFGS,Difference
term,,,
count:Intercept,-4.019273,-4.036223,0.016951
count:group,-0.693593,-0.684108,-0.009485
count:grna_n_nonzero,0.005191,0.004686,0.000505
count:grna_n_umis,-0.008609,-0.000477,-0.008132
count:response_n_umis,0.415146,0.417534,-0.002387
count:percent_mt,-0.399569,-0.421696,0.022127
count:inlet_inlet2,-0.122185,-0.106422,-0.015763
count:inlet_inlet3,-0.463959,-0.487151,0.023192
log(theta),-0.869455,-0.790097,-0.079358


In [13]:
summary_test = res_hybrid["FOXP2"].summary()
# summary_test.index = [f"count: {col}" for col in X.columns.to_list()]+[f"zero: {col}" for col in X.columns.to_list()]  + ["log_theta"]
summary_test

,Estimate,Std. Error,z value,Pr(>|z|)
term,,,,
count:Intercept,-6.380692,1.146487,-5.565431,2.615059e-08
count:group,-0.693184,0.100372,-6.906146,4.979980e-12
count:grna_n_nonzero,-0.010608,0.007983,-1.328792,1.839167e-01
count:grna_n_umis,0.083616,0.068078,1.228237,2.193580e-01
count:response_n_umis,0.679467,0.116167,5.849049,4.943920e-09
count:percent_mt,-0.526237,0.106521,-4.940233,7.802922e-07
count:inlet_inlet2,-0.367501,0.154003,-2.386332,1.701739e-02
count:inlet_inlet3,-0.576740,0.167437,-3.444518,5.720792e-04
zero:Intercept,0.754439,10.055932,0.075024,9.401954e-01


In [8]:
start_time = time.time()
res_lbfgs = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,
    model="mzinb",
    fit_cfg=FitConfig(method="lbfgs", verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
    lrt_target="group"
)

elapsed_time_lbfgs = time.time() - start_time

In [4]:
res_lbfgs["FOXP2"].lrt()

,LR,df,p_value,ll_full,ll_reduced
term,,,,,
group,115.852539,2,6.965315e-26,-4387.231934,-4445.158203


In [ ]:
adata = ad.read_h5ad("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/Results/cropseq_matrix_v2/obj.h5ad")
covariates_df = pd.read_csv("/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/Results/DEG_Analysis/DEG_covariates.csv", index_col=0)


In [15]:
gene_targets = ["FOXP2","CALB2","MEF2C","SEMA6D","RARA","SOX10"]
for gene_target in gene_targets:
    covariates_df["expression"] = adata[:,gene_target].X.toarray().flatten()
    df_sub = covariates_df[(covariates_df['gRNA'].str.contains(gene_target)) | (covariates_df['gRNA'].str.contains("non-targeting"))]
    df_sub["group"] = np.where(df_sub['gRNA'].str.contains(gene_target), 1, 0)
    common = adata.obs_names.intersection(df_sub.index)
    adata_sub = adata[common, :].copy()
    df_sub = df_sub.loc[common].copy()   
    predictors = ["group", "grna_n_nonzero", "grna_n_umis", "response_n_umis", "percent_mt", "inlet"]
    X = df_sub[predictors].copy()

    log_cols = ["grna_n_umis", "response_n_umis"]
    X[log_cols] = X[log_cols].apply(lambda s: np.log1p(pd.to_numeric(s, errors="coerce")))

    num_cols = ["group", "grna_n_nonzero", "percent_mt"]
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X["inlet"] = X["inlet"].astype("category")
    X = pd.get_dummies(X, columns=["inlet"], drop_first=True)

    X.insert(0, "Intercept", 1.0)

    mask = X.notna().all(axis=1)
    adata_sub = adata_sub[mask.values, :].copy()
    X = X.loc[mask].astype(np.float32)
    res_lbfgs = fit_glm(
    adata_sub,
    genes=[gene_target],
    X_design=X,
    model="mzinb",
    fit_cfg=FitConfig(method="lbfgs", verbose=False),
    glm_cfg=GLMConfig(offset=False),
    device="cpu",
    lrt_target="group"
    )
    print(f"Results for gene: {gene_target}")
    print(res_lbfgs[gene_target].lrt())

/tmp/ipykernel_836023/1625336846.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["group"] = np.where(df_sub['gRNA'].str.contains(gene_target), 1, 0)


Results for gene: FOXP2
               LR  df       p_value      ll_full   ll_reduced
term                                                         
group  115.852539   2  6.965315e-26 -4387.231934 -4445.158203


/tmp/ipykernel_836023/1625336846.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["group"] = np.where(df_sub['gRNA'].str.contains(gene_target), 1, 0)
/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/src/deg_zinb/inference.py:91: RuntimeWarning: divide by zero encountered in divide
  z = theta_hat / se


Results for gene: CALB2
             LR  df   p_value     ll_full  ll_reduced
term                                                 
group  6.665161   2  0.035701 -970.804871 -974.137451


/tmp/ipykernel_836023/1625336846.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["group"] = np.where(df_sub['gRNA'].str.contains(gene_target), 1, 0)
/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/src/deg_zinb/inference.py:91: RuntimeWarning: divide by zero encountered in divide
  z = theta_hat / se


...

Results for gene: SEMA6D
             LR  df   p_value       ll_full    ll_reduced
term                                                     
group  1.320312   2  0.516771 -19300.414062 -19301.074219


/tmp/ipykernel_836023/1625336846.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["group"] = np.where(df_sub['gRNA'].str.contains(gene_target), 1, 0)
/proj/hyejunglab/cropseq/Alejandro/Mint/DEG_Analysis/deg_zinb/src/deg_zinb/inference.py:91: RuntimeWarning: divide by zero encountered in divide
  z = theta_hat / se


Results for gene: RARA
             LR  df   p_value      ll_full   ll_reduced
term                                                   
group  9.227295   2  0.009916 -1009.094604 -1013.708252


/tmp/ipykernel_836023/1625336846.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["group"] = np.where(df_sub['gRNA'].str.contains(gene_target), 1, 0)


Results for gene: SOX10
             LR  df   p_value     ll_full  ll_reduced
term                                                 
group  0.466537   2  0.791941 -124.929283 -125.162552
